In [ ]:
import os
import cv2
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Conv2D,MaxPooling2D,Flatten,BatchNormalization,Dropout


In [ ]:
# generators
train_ds=keras.utils.image_dataset_from_directory(
    directory='train',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256,256)
)

validate_ds=keras.utils.image_dataset_from_directory(
    directory='test',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256,256)
)


In [ ]:
# Normalize
def process(image,label):
    image=tf.cast(image/255,tf.float32)
    return image,label

train_ds=train_ds.map(process)
validate_ds=validate_ds.map(process)

In [ ]:
# CNN Model
model=Sequential()
model.add(Conv2D(32,kernel_size=(3,3),padding='valid',activation='relu',input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))
model.add(Conv2D(64,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))
model.add(Conv2D(128,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))
model.add(Flatten())
model.add(Dense(128,activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(64,activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(1,activation='sigmoid'))

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], color='red', label='train')
plt.plot(history.history['val_accuracy'], color='blue', label='validation')
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
 
# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], color='red', label='train')
plt.plot(history.history['val_loss'], color='blue', label='validation')
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()
 
# ==========================================
# 7. INFERENCE (PREDICTING ON NEW IMAGES)
# ==========================================
# Replace path with your actual test image path (e.g., 'cat.jpg' or 'dog.jpg')
image_path = '/content/cat.jpg'
 
if os.path.exists(image_path):
    test_img = cv2.imread(image_path)
    # Convert BGR (OpenCV default) to RGB for correct matplotlib plotting
    test_img_rgb = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)
    plt.imshow(test_img_rgb)
    plt.axis('off')
    plt.show()
 
    # Preprocess the image to fit model expectations
    test_img_resized = cv2.resize(test_img, (256, 256))
    test_input = test_img_resized.reshape((1, 256, 256, 3))
 
    # Scale image pixels if the training pipeline scales them
    test_input = test_input / 255.0
 
    # Prediction
    prediction = model.predict(test_input)
    print(f"Raw Prediction Value: {prediction[0][0]}")
 
    if prediction[0][0] < 0.5:
        print("Prediction: It's a CAT!")
    else:
        print("Prediction: It's a DOG!")
else:
    print(f"Image not found at {image_path}. Please check your path.")
 

In [ ]:
history=model.fit(train_ds,epochs=10,validation_data=validate_ds)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], color='red', label='train')
plt.plot(history.history['val_accuracy'], color='blue', label='validation')
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
 

In [ ]:
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], color='red', label='train')
plt.plot(history.history['val_loss'], color='blue', label='validation')
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
cat_image_path = 'cat.jpg'
 
if os.path.exists(cat_image_path):
    cat_test_img = cv2.imread(cat_image_path)
    # Convert BGR (OpenCV default) to RGB for correct matplotlib plotting
    cat_test_img_rgb = cv2.cvtColor(cat_test_img, cv2.COLOR_BGR2RGB)
    plt.imshow(cat_test_img_rgb)
    plt.axis('off')
    plt.show()

In [ ]:
dog_image_path = 'dog.jpg'
if os.path.exists(dog_image_path):
    dog_test_img = cv2.imread(dog_image_path)
    # Convert BGR (OpenCV default) to RGB for correct matplotlib plotting
    dog_test_img_rgb = cv2.cvtColor(dog_test_img, cv2.COLOR_BGR2RGB)
    plt.imshow(dog_test_img_rgb)
    plt.axis('off')
    plt.show()

In [ ]:
cat_test_img_resized = cv2.resize(cat_test_img, (256, 256))
cat_test_input = cat_test_img_resized.reshape((1, 256, 256, 3))
 
    # Scale image pixels if the training pipeline scales them
cat_test_input = cat_test_input / 255.0
 
    # Prediction
cat_prediction = model.predict(cat_test_input)
print(f"Raw Prediction Value: {cat_prediction[0][0]}")

In [ ]:
dog_test_img_resized = cv2.resize(dog_test_img, (256, 256))
dog_test_input = dog_test_img_resized.reshape((1, 256, 256, 3))
 
    # Scale image pixels if the training pipeline scales them
dog_test_input = dog_test_input / 255.0
 
    # Prediction
dog_prediction = model.predict(dog_test_input)
print(f"Raw Prediction Value: {dog_prediction[0][0]}")

In [40]:
model.save("cat_dog_model.keras")